In [2]:
# Run this once in your terminal:
# pip install mediacloud pandas

import mediacloud.api
import pandas as pd
from datetime import datetime

# Paste your Media Cloud API key here
API_KEY = "164033ccca18dc3302221c744f3cc92b7fca5ecd"

# Canada National collection — contains 1000s of Canadian news sources
CANADA_COLLECTION = 34411583

In [3]:
def search_articles(topic, start_date, end_date):
    """
    Pull articles from Media Cloud about a topic.
    start_date / end_date format: 'YYYY-MM-DD'
    Returns a list of article dictionaries.
    """
    # Convert date strings to date objects (required by the API)
    start_date = datetime.strptime(start_date, "%Y-%m-%d").date()
    end_date = datetime.strptime(end_date, "%Y-%m-%d").date()

    # Connect to the Media Cloud Search API
    mc = mediacloud.api.SearchApi(API_KEY)

    all_stories = []
    pagination_token = None

    while True:
        # Fetch one page of results
        page, pagination_token = mc.story_list(
            topic,
            start_date=start_date,
            end_date=end_date,
            collection_ids=[CANADA_COLLECTION],
            pagination_token=pagination_token
        )
        all_stories += page

        print(f"  Fetched {len(all_stories)} articles so far...")

        if not pagination_token:
            break   # no more pages left

    return all_stories

In [4]:
# ── Change these three lines for YOUR topic ──────────
TOPIC = "Lacrosse"
START = "2026-01-01"
END   = "2026-12-31"
# ─────────────────────────────────────────────────────

print(f"Searching for: {TOPIC}")
results = search_articles(TOPIC, START, END)
print(f"Total found: {len(results)} articles")

# Turn into a table and keep the useful columns
df = pd.DataFrame(results)

# Only keep columns that exist in the results
keep = ["publish_date", "title", "url", "media_name", "language"]
keep = [c for c in keep if c in df.columns]
df = df[keep]

# Save to a CSV file
df.to_csv("my_articles.csv", index=False)
print("✅ Saved to my_articles.csv")
print(df.head(5))   # preview the first 5 rows

Searching for: Lacrosse
  Fetched 1000 articles so far...
  Fetched 2000 articles so far...
  Fetched 2866 articles so far...
Total found: 2866 articles
✅ Saved to my_articles.csv
  publish_date                                              title  \
0   2026-07-03  Phillips on the ALCS rematch between the Jays ...   
1   2026-07-03  Strome on the return of Ovechkin, Capitals add...   
2   2026-07-03  TSN Mornings: Jason deVos says Canada’s tactic...   
3   2026-07-03  Sutton on what Canada needs to do to upset Mor...   
4   2026-07-03                            WATCH LIVE: Arabic Feed   

                                                 url media_name language  
0  https://www.tsn.ca/radio/toronto-1050/article/...     tsn.ca       en  
1  https://www.tsn.ca/radio/toronto-1050/article/...     tsn.ca       en  
2  https://www.tsn.ca/radio/ottawa-1200/article/t...     tsn.ca       en  
3  https://www.tsn.ca/radio/toronto-1050/article/...     tsn.ca       en  
4  https://www.tsn.ca/wires/ar